# Análise de Dados do ENEM 2021 no Amapá
Estudo estatístico onde serão abordadas algumas variáveis sobre o ENEM 2021 no Estado do Amapá, fazendo comparação entre dados de alunos de escolas públicas e privadas.  
Para esse estudo, será utilizado um dataset pré-tratado, relativo às provas do ENEM, realizadas no Amapá, que por sua vez, foi retirado dos [microdados](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/enem) do ENEM 2021.

## Preparação, Organização e Estruturação dos Dados
### Importando o pandas

In [ ]:
import pandas as pd

In [ ]:
pd.set_option("display.max_columns", 100)

In [ ]:
df = pd.read_csv("/kaggle/input/enem-2021-amap/enem_2021_ap.csv",
                 encoding="iso-8859-1")
df.head()

In [ ]:
df.shape

## Valores nulos
verificando valores nulos das principais variáveis do estudo.

In [ ]:
df[["NOTA_CN",	"NOTA_CH",	"NOTA_LC",	"NOTA_MT", "NOTA_REDACAO", "TP_ESCOLA",
           "FAIXA_ETARIA", "Q001",	"Q002",	"Q003",	"Q004",	"Q005",
           "Q006",	"Q007",	"Q008",	"Q009",	"Q010",	"Q011",	"Q012",	"Q013",
           "Q014",	"Q015",	"Q016",	"Q017",	"Q018",	"Q019",	"Q020",	"Q021",
           "Q022",	"Q023",	"Q024",	"Q025"]].isnull().sum()

# Identificação da População e das Variáveis
A população de estudo será dividida em dois grupos de vestibulandos por tipo de escola:
* públicas
* privadas

In [ ]:
df.TP_ESCOLA.value_counts()

### Removendo vestibulandos que não responderam a qual tipo de escola pertencem

In [ ]:
esc_invalida = df[df.TP_ESCOLA == "nao_respondeu"].index
df_estudo = df.copy().drop(esc_invalida)

In [ ]:
nota_final = ["NOTA_CN", "NOTA_CH",	"NOTA_LC",	"NOTA_MT", "NOTA_REDACAO"]
df_estudo.insert(20, "NOTA_FINAL", df_estudo[nota_final].sum(axis=1) / 5)

In [ ]:
df_estudo.FAIXA_ETARIA = df_estudo.FAIXA_ETARIA.replace(
    {1:	"Menor de 17 anos",
    2:	"17 anos",
    3:	"18 anos",
    4:	"19 anos",
    5:	"20 anos",
    6:	"21 anos",
    7:	"22 anos",
    8:	"23 anos",
    9:	"24 anos",
    10:	"25 anos",
    11:	"Entre 26 e 30 anos",
    12:	"Entre 31 e 35 anos",
    13:	"Entre 36 e 40 anos",
    14:	"Entre 41 e 45 anos",
    15:	"Entre 46 e 50 anos",
    16:	"Entre 51 e 55 anos",
    17:	"Entre 56 e 60 anos",
    18:	"Entre 61 e 65 anos",
    19:	"Entre 66 e 70 anos",
    20:	"Maior de 70 anos"}
)

df_estudo.head()

In [ ]:
df_estudo.shape

In [ ]:
df_estudo.TP_ESCOLA.value_counts()

O total de vestibulandos de escolas públicas e privadas é de 3439. Desses, 2961 matriculados em escolas públicas e 478 em escolas privadas.

**Escola Pública:**

In [ ]:
esc_publica = df_estudo.copy().query("TP_ESCOLA == 'publica'").drop("TP_ESCOLA", axis=1)

esc_publica.shape

**Escola Privada:**

In [ ]:
esc_privada = df_estudo.copy().query("TP_ESCOLA == 'privada'").drop("TP_ESCOLA", axis=1)

esc_privada.shape

# Descrição dos Dados
## Frequências absoluta, percentual e percentual relativa
Frequências absoluta, percentual e percentual relativa da faixa etária de idades dos vestibulandos.

**Escolas Públicas:**

In [ ]:
abs_pub = pd.DataFrame.from_dict(esc_publica.FAIXA_ETARIA.value_counts()).rename(columns={"FAIXA_ETARIA": "ABSOLUTA"})

rel_pub = pd.DataFrame.from_dict(round(abs_pub / abs_pub.sum(), 4)).rename(columns={"ABSOLUTA": "RELATIVA"})

perc_pub = pd.DataFrame.from_dict(round(rel_pub * 100, 4)).rename(columns={"RELATIVA": "PERCENTUAL_RELATIVA"})

In [ ]:
df_freq_pub = pd.concat([abs_pub, rel_pub, perc_pub], axis=1)
df_freq_pub

Percebemos que a maior parte dos valores para as escolas públicas está concetrada entre as faixas de 17 a 20 anos

**Escolas Privadas:**

In [ ]:
abs_priv = pd.DataFrame.from_dict(esc_privada.FAIXA_ETARIA.value_counts()).rename(columns={"FAIXA_ETARIA": "ABSOLUTA"})

rel_priv = pd.DataFrame.from_dict(round(abs_priv / abs_priv.sum(), 4)).rename(columns={"ABSOLUTA": "RELATIVA"})

perc_priv = pd.DataFrame.from_dict(round(rel_priv * 100, 4)).rename(columns={"RELATIVA": "PERCENTUAL_RELATIVA"})

In [ ]:
df_freq_priv = pd.concat([abs_priv, rel_priv, perc_priv], axis=1)
df_freq_priv

Para as escolas privadas, idades entre 17 e 18 anos predominam, muito dentro do que se encontra para idade do ensino médio regular.  
Além disso, no escopo do conjunto de dados a maior ocorrência de idades foram de pessoas de até 19 anos.

### Visualização de frequências

In [ ]:
import plotly.express as px

**Notas:**  
Frequência de notas finais do enem, em relação ao percentual em que as mesmas aparecem, separadas pelo tipo da escola.

In [ ]:
graph_freq_notas = px.histogram(df_estudo, x="NOTA_FINAL", height=700,
                                marginal="rug", histnorm="percent",
                                barmode="group", color="TP_ESCOLA")
graph_freq_notas.update_layout(bargap=0)

Percebemos que as notas das escolas públicas estão distribuidas assimétricamente à direita, enquanto que, as de escolas particulares, encontram-se mais uniformemente distribuídas. Ou seja, para as escolas públicas, quando o valor da nota aumenta, o número de ocorrências diminui.

**Faixas Etárias:**  
Frequência de faixa etárias dos vestibulandos que fizeram a prova do ENEM 2021.

In [ ]:
fx_eta_pub = px.bar(df_freq_pub, x="ABSOLUTA", text_auto=True)
fx_eta_pub.show()

In [ ]:
fx_eta_priv = px.bar(df_freq_priv, x="ABSOLUTA", text_auto=True)
fx_eta_priv.show()

A variabilidade de participantes vestibulandos em diferentes faixa etárias de escolas públicas é maior, em relação às escolas privadas. Além disso, tanto para escolas públicas, quanto para privadas, a maior ocorrência foram de alunos entre 18 e 17 anos.

## Medidas de Tendência Central  
Média, moda e mediana de escolas públicas e privadas.

**Média:**

In [ ]:
mean_pub = round(esc_publica.NOTA_FINAL.mean(), 2)
mean_priv = round(esc_privada["NOTA_FINAL"].mean(), 2)

**Moda:**

In [ ]:
mod_pub = round(esc_publica.NOTA_FINAL.mode(), 2).values
mod_priv = round(esc_privada["NOTA_FINAL"].mode(), 2).values

**Mediana:**

In [ ]:
medn_pub = round(esc_publica.NOTA_FINAL.median(), 2)
medn_priv = round(esc_privada["NOTA_FINAL"].median(), 2)

In [ ]:
data_cent = {
    "MEDIA": [mean_pub, mean_priv],
    "MODA": [mod_pub, mod_priv],
    "MEDIANA": [medn_pub, medn_priv]
}

In [ ]:
df_med_cent = pd.DataFrame(data_cent, index=["ESCOLA_PUBLICA", "ESCOLA_PRIVADA"])
df_med_cent

In [ ]:
round((mean_priv / mean_pub) * 100, 2)

A partir da tabela, podemos concluir que a média das notas finais das escolas privadas é 118,1% maior, em relação as das escolas públicas.  
É possível notar também que, ambos os tipos de escolas apresentaram 3 modas cada.

## Medidas de dispersão ou variação
Será verificado o grau de variação das notas finais com relação a média.

**Amplitude:**

In [ ]:
ampt_pub = esc_publica.NOTA_FINAL.max() - esc_publica["NOTA_FINAL"].min()
ampt_priv = esc_privada["NOTA_FINAL"].max() - esc_privada.NOTA_FINAL.min()

**Desvio Padrão:**

In [ ]:
std_pub = round(esc_publica.NOTA_FINAL.std(), 2)
std_priv = round(esc_privada.NOTA_FINAL.std(), 2)

**Variância:**

In [ ]:
var_pub = round(esc_publica.NOTA_FINAL.var(ddof=0), 2)
var_priv = round(esc_privada.NOTA_FINAL.var(ddof=0), 2)

**Desvio Absoluto Médio:**

In [ ]:
mad_pub = round(esc_publica.NOTA_FINAL.mad(), 2)
mad_priv = round(esc_privada.NOTA_FINAL.mad(), 2)

In [ ]:
data_disp_var = {
    "AMPLITUDE": [ampt_pub, ampt_priv],
    "DESVIO_PADRAO": [std_pub, std_priv],
    "VARIANCIA": [var_pub, var_priv],
    "DESVIO_ABS_MEDIO": [mad_pub, mad_priv]
}

In [ ]:
df_disp_var = pd.DataFrame(data_disp_var, index=["ESCOLA_PUBLICA", "ESCOLA_PRIVADA"])
df_disp_var

Olhando para o desvio padrão, percebemos que as notas das escolas privadas estão melhores distribuídas em torno da média.  
Além disso, a variância nos mostra que os dados das escola públicas estão mais condensados em torno do valor central.

## Medidas de Posição

Fractis:

In [ ]:
q1_priv = esc_privada.NOTA_FINAL.quantile(q=0.25)
q2_priv = esc_privada.NOTA_FINAL.median()
q3_priv = esc_privada.NOTA_FINAL.quantile(q=0.75)
q4_priv = esc_privada.NOTA_FINAL.max()

In [ ]:
q1_pub = esc_publica.NOTA_FINAL.quantile(q=0.25)
q2_pub = esc_publica.NOTA_FINAL.quantile(q=0.5)
q3_pub = esc_publica.NOTA_FINAL.quantile(q=0.75)
q4_pub = esc_publica.NOTA_FINAL.quantile(q=1)

In [ ]:
iqr_pub = q3_pub - q1_pub
iqr_priv = q3_priv - q1_priv

In [ ]:
data_med_pos = {
    "Q1": [q1_pub, q1_priv],
    "Q2": [q2_pub, q2_priv],
    "Q3": [q3_pub, q3_priv],
    "Q4": [q4_pub, q4_priv],
    "IQR": [iqr_pub, iqr_priv]
}

In [ ]:
df_med_pos = pd.DataFrame(data_med_pos, index=["ESCOLA_PUBLICA", "ESCOLA_PRIVADA"])
df_med_pos

### Outliers

In [ ]:
px.box(data_frame=df_estudo, y="NOTA_FINAL", width=500, height=600,
       color="TP_ESCOLA")

Há diversas ocorrências de dados discrepantes para as escolas públicas, enquanto que, para escolas privadas, o mesmo não acontece. Com isso, subentende-se que, as notas das escolas privadas encontram certa consistência entre si.  
Também é possível notar nas notas mais altas, uam concentração percentual maior para escolas privadas.

## Comparação Entre os Questionários dos Alunos de Escolas Públicas e Privadas
Para esse módulo, será criado um novo dataset, apenas com as variáveis a serem visualizadas.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
df_plot = df_estudo.copy()[["TP_ESCOLA", "Q001", "Q002", "Q003", "Q004", "Q005",
                            "Q006", "Q007", "Q008", "Q009", "Q010", "Q011",
                            "Q012", "Q013", "Q014", "Q015", "Q016", "Q017",
                            "Q018", "Q019", "Q020", "Q021", "Q022", "Q023",
                            "Q024", "Q025"]]
df_plot.head()

### Fazendo substituição dos valores nos registros

Será feito uma substituição dos valores das linhas (registros), para que se facilite a compreensão dos mesmos.

In [ ]:
df_plot.Q001 = df_plot.Q001.replace(
    {"A": "Nunca estudou",
     "B": "Não completou a 4ª série/5º ano do Ensino Fundamental",
     "C": "Completou a 4ª série/5º ano, mas não completou a 8ª série/9º ano do Ensino Fundamental",
     "D": "Completou a 8ª série/9º ano do Ensino Fundamental, mas não completou o Ensino Médio",
     "E": "Completou o Ensino Médio, mas não completou a Faculdade",
     "F": "Completou a Faculdade, mas não completou a Pós-graduação",
     "G": "Completou a Pós-graduação",
     "H": "Não sei"})

In [ ]:
df_plot.Q002 = df_plot.Q002.replace(
   {"A": "Nunca estudou",
     "B": "Não completou a 4ª série/5º ano do Ensino Fundamental",
     "C": "Completou a 4ª série/5º ano, mas não completou a 8ª série/9º ano do Ensino Fundamental",
     "D": "Completou a 8ª série/9º ano do Ensino Fundamental, mas não completou o Ensino Médio",
     "E": "Completou o Ensino Médio, mas não completou a Faculdade",
     "F": "Completou a Faculdade, mas não completou a Pós-graduação",
     "G": "Completou a Pós-graduação",
     "H": "Não sei"})

In [ ]:
df_plot.Q006 = df_plot.Q006.replace(
   {"A":	"Nenhuma Renda",
   "B": "Até R$ 1.100,00",
    "C":	"De R$ 1.100,01 até R$ 1.650,00",
    "D":	"De R$ 1.650,01 até R$ 2.200,00",
    "E":	"De R$ 2.200,01 até R$ 2.750,00",
    "F":	"De R$ 2.750,01 até R$ 3.300,00",
    "G":	"De R$ 3.300,01 até R$ 4.400,00",
    "H":	"De R$ 4.400,01 até R$ 5.500,00",
    "I":	"De R$ 5.500,01 até R$ 6.600,00",
    "J":	"De R$ 6.600,01 até R$ 7.700,00",
    "K":	"De R$ 7.700,01 até R$ 8.800,00",
    "L":	"De R$ 8.800,01 até R$ 9.900,00",
    "M":	"De R$ 9.900,01 até R$ 11.000,00",
    "N":	"De R$ 11.000,01 até R$ 13.200,00",
    "O":	"De R$ 13.200,01 até R$ 16.500,00",
    "P":	"De R$ 16.500,01 até R$ 22.000,00",
    "Q":	"Acima de R$ 22.000,00"})

In [ ]:
df_plot.Q008 = df_plot.Q008.replace(
   {"A":	"Não",
   "B":	"Sim, um",
   "C":	"Sim, dois",
   "D":	"Sim, três",
   "E":	"Sim, quatro ou mais"})

In [ ]:
df_plot.Q009 = df_plot.Q009.replace(
   {"A":	"Não",
   "B":	"Sim, um",
   "C":	"Sim, dois",
   "D":	"Sim, três",
   "E":	"Sim, quatro ou mais"})

In [ ]:
df_plot.Q012 = df_plot.Q012.replace(
   {"A":	"Não",
   "B":	"Sim, uma",
   "C":	"Sim, duas",
   "D":	"Sim, três",
   "E":	"Sim, quatro ou mais"})

In [ ]:
df_plot.Q014 = df_plot.Q014.replace(
   {"A":	"Não",
   "B":	"Sim, uma",
   "C":	"Sim, duas",
   "D":	"Sim, três",
   "E":	"Sim, quatro ou mais"})

In [ ]:
df_plot.Q019 = df_plot.Q019.replace(
   {"A":	"Não",
   "B":	"Sim, uma",
   "C":	"Sim, duas",
   "D":	"Sim, três",
   "E":	"Sim, quatro ou mais"})

In [ ]:
df_plot.Q022 = df_plot.Q022.replace(
   {"A":	"Não",
   "B":	"Sim, um",
   "C":	"Sim, dois",
   "D":	"Sim, três",
   "E":	"Sim, quatro ou mais"})

In [ ]:
df_plot.Q024 = df_plot.Q024.replace(
   {"A":	"Não",
   "B":	"Sim, um",
   "C":	"Sim, dois",
   "D":	"Sim, três",
   "E":	"Sim, quatro ou mais"})

In [ ]:
df_plot.Q025 = df_plot.Q025.replace(
   {"A":	"Não",
   "B":	"Sim"})

In [ ]:
df_plot.head()

In [ ]:
quest = {"Q001": "Até que série seu pai, ou o homem responsável por você, estudou?",
 "Q002": "Até que série sua mãe, ou a mulher responsável por você, estudou?",
 "Q005": "Incluindo você, quantas pessoas moram atualmente em sua residência?",
 "Q006": "Qual é a renda mensal de sua família? (Some a sua renda com a dos seus familiares.)",
 "Q008": "Na sua residência tem banheiro?",
 "Q009": "Na sua residência tem quartos para dormir?",
 "Q012": "Na sua residência tem geladeira?",
 "Q014": "Na sua residência tem máquina de lavar roupa? (o tanquinho NÃO deve ser considerado)",
 "Q019": "Na sua residência tem televisão em cores?",
 "Q022": "Na sua residência tem telefone celular?",
 "Q024": "Na sua residência tem computador?",
 "Q025": "Na sua residência tem acesso à Internet?"}

### Visualização gráfica entre vestibulandos de escolas públicas e privadas

In [ ]:
q001_pub = df_plot.query("TP_ESCOLA == 'publica'")["Q001"].value_counts()
label_pub = q001_pub.index
value_pub = q001_pub.values

q001_priv = df_plot.query("TP_ESCOLA == 'privada'")["Q001"].value_counts()
label_priv = q001_priv.index
value_priv = q001_priv.values

graph_q001 = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'},
                                          {'type':'domain'}]],
                  subplot_titles=["Escolas Públicas", "Escolas Privadas"])

graph_q001.add_trace(go.Pie(labels=label_pub, values=value_pub), 1, 1)
graph_q001.add_trace(go.Pie(labels=label_priv, values=value_priv), 1, 2)

graph_q001.update_layout(title_text=quest["Q001"])
graph_q001.update_layout(uniformtext_minsize=14, uniformtext_mode='hide')

Para ambos os tipos de escolas, o que predomina são pais que completaram o ensino médio, mas não a faculdade. Os dados começam a ficar discrepantes quando se compara os que completaram a faculdade e a pós graduação, por exemplo.

In [ ]:
q002_pub = df_plot.query("TP_ESCOLA == 'publica'")["Q002"].value_counts()
label_pub = q002_pub.index
value_pub = q002_pub.values

q002_priv = df_plot.query("TP_ESCOLA == 'privada'")["Q002"].value_counts()
label_priv = q002_priv.index
value_priv = q002_priv.values

graph_q002 = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'},
                                          {'type':'domain'}]],
                  subplot_titles=["Escolas Públicas", "Escolas Privadas"])

graph_q002.add_trace(go.Pie(labels=label_pub, values=value_pub), 1, 1)
graph_q002.add_trace(go.Pie(labels=label_priv, values=value_priv), 1, 2)

graph_q002.update_layout(title_text=quest["Q002"])
graph_q002.update_layout(uniformtext_minsize=14, uniformtext_mode='hide')

Já para as mães, a maior concentração está na faixa dos 40%, porém, com valores totalmente diferentes. As de escolas públicas, completaram apenas o ensino médio, enquanto que as de escolas privadas, completaram a pós graduação.

In [ ]:
q005_pub = df_plot.query("TP_ESCOLA == 'publica'")["Q005"].value_counts()
label_pub = q005_pub.index
value_pub = q005_pub.values

q005_priv = df_plot.query("TP_ESCOLA == 'privada'")["Q005"].value_counts()
label_priv = q005_priv.index
value_priv = q005_priv.values

graph_q005 = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'},
                                          {'type':'domain'}]],
                  subplot_titles=["Escolas Públicas", "Escolas Privadas"])

graph_q005.add_trace(go.Pie(labels=label_pub, values=value_pub), 1, 1)
graph_q005.add_trace(go.Pie(labels=label_priv, values=value_priv), 1, 2)

graph_q005.update_layout(title_text=quest["Q005"])
graph_q005.update_layout(uniformtext_minsize=14, uniformtext_mode='hide')

Para ambos, a maioria reside em residências com total de habitantes entre 3 e 6 pessoas.

In [ ]:
q006_pub = df_plot.query("TP_ESCOLA == 'publica'")["Q006"].value_counts()
label_pub = q006_pub.index
value_pub = q006_pub.values

q006_priv = df_plot.query("TP_ESCOLA == 'privada'")["Q006"].value_counts()
label_priv = q006_priv.index
value_priv = q006_priv.values

graph_q006 = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'},
                                          {'type':'domain'}]],
                  subplot_titles=["Escolas Públicas", "Escolas Privadas"])

graph_q006.add_trace(go.Pie(labels=label_pub, values=value_pub), 1, 1)
graph_q006.add_trace(go.Pie(labels=label_priv, values=value_priv), 1, 2)

graph_q006.update_layout(title_text=quest["Q006"])
graph_q006.update_layout(uniformtext_minsize=14, uniformtext_mode='hide')

A renda mensal familiar dos vestibulandos de escolas privadas está bem distribuída entre valores de R\\$1100,00 até R\\$13200,00. O mesmo não se vê para os de escolas públicas, onde a concentração está na renda de até R\\$1100,00, porém, encontramos ainda 7,36% dos vestibulandos com nenhuma renda familiar.

In [ ]:
q008_pub = df_plot.query("TP_ESCOLA == 'publica'")["Q008"].value_counts()
label_pub = q008_pub.index
value_pub = q008_pub.values

q008_priv = df_plot.query("TP_ESCOLA == 'privada'")["Q008"].value_counts()
label_priv = q008_priv.index
value_priv = q008_priv.values

graph_q008 = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'},
                                          {'type':'domain'}]],
                  subplot_titles=["Escolas Públicas", "Escolas Privadas"])

graph_q008.add_trace(go.Pie(labels=label_pub, values=value_pub), 1, 1)
graph_q008.add_trace(go.Pie(labels=label_priv, values=value_priv), 1, 2)

graph_q008.update_layout(title_text=quest["Q008"])
graph_q008.update_layout(uniformtext_minsize=14, uniformtext_mode='hide')

Mesmo que apenas 18 de um total de 2961, representando 0.68% dos dados, ainda vemos residências de vestibulandos em que não há banheiro.

In [ ]:
q009_pub = df_plot.query("TP_ESCOLA == 'publica'")["Q009"].value_counts()
label_pub = q009_pub.index
value_pub = q009_pub.values

q009_priv = df_plot.query("TP_ESCOLA == 'privada'")["Q009"].value_counts()
label_priv = q009_priv.index
value_priv = q009_priv.values

graph_q009 = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'},
                                          {'type':'domain'}]],
                  subplot_titles=["Escolas Públicas", "Escolas Privadas"])

graph_q009.add_trace(go.Pie(labels=label_pub, values=value_pub), 1, 1)
graph_q009.add_trace(go.Pie(labels=label_priv, values=value_priv), 1, 2)

graph_q009.update_layout(title_text=quest["Q009"])
graph_q009.update_layout(uniformtext_minsize=14, uniformtext_mode='hide')

50,4% das residências de vestibulandos de escolas públicas tem dois quartos. Ainda assim, encontramos 1,38% das mesmas com nenhum quarto.

In [ ]:
q012_pub = df_plot.query("TP_ESCOLA == 'publica'")["Q012"].value_counts()
label_pub = q012_pub.index
value_pub = q012_pub.values

q012_priv = df_plot.query("TP_ESCOLA == 'privada'")["Q012"].value_counts()
label_priv = q012_priv.index
value_priv = q012_priv.values

graph_q012 = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'},
                                          {'type':'domain'}]],
                  subplot_titles=["Escolas Públicas", "Escolas Privadas"])

graph_q012.add_trace(go.Pie(labels=label_pub, values=value_pub), 1, 1)
graph_q012.add_trace(go.Pie(labels=label_priv, values=value_priv), 1, 2)

graph_q012.update_layout(title_text=quest["Q012"])
graph_q012.update_layout(uniformtext_minsize=14, uniformtext_mode='hide')

Para ambos vemos que a maioria percentual de vestibulandos possuem ao menos uma geladeira em sua residência. Ainda assim, um número relativamente grande (4,39%) dos vestibulandos de escolas públicas não possuem nenhuma.

In [ ]:
q014_pub = df_plot.query("TP_ESCOLA == 'publica'")["Q014"].value_counts()
label_pub = q014_pub.index
value_pub = q014_pub.values

q014_priv = df_plot.query("TP_ESCOLA == 'privada'")["Q014"].value_counts()
label_priv = q014_priv.index
value_priv = q014_priv.values

graph_q014 = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'},
                                          {'type':'domain'}]],
                  subplot_titles=["Escolas Públicas", "Escolas Privadas"])

graph_q014.add_trace(go.Pie(labels=label_pub, values=value_pub), 1, 1)
graph_q014.add_trace(go.Pie(labels=label_priv, values=value_priv), 1, 2)

graph_q014.update_layout(title_text=quest["Q014"])
graph_q014.update_layout(uniformtext_minsize=14, uniformtext_mode='hide')

In [ ]:
q019_pub = df_plot.query("TP_ESCOLA == 'publica'")["Q019"].value_counts()
label_pub = q019_pub.index
value_pub = q019_pub.values

q019_priv = df_plot.query("TP_ESCOLA == 'privada'")["Q019"].value_counts()
label_priv = q019_priv.index
value_priv = q019_priv.values

graph_q019 = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'},
                                          {'type':'domain'}]],
                  subplot_titles=["Escolas Públicas", "Escolas Privadas"])

graph_q019.add_trace(go.Pie(labels=label_pub, values=value_pub), 1, 1)
graph_q019.add_trace(go.Pie(labels=label_priv, values=value_priv), 1, 2)

graph_q019.update_layout(title_text=quest["Q019"])
graph_q019.update_layout(uniformtext_minsize=14, uniformtext_mode='hide')

In [ ]:
q022_pub = df_plot.query("TP_ESCOLA == 'publica'")["Q022"].value_counts()
label_pub = q022_pub.index
value_pub = q022_pub.values

q022_priv = df_plot.query("TP_ESCOLA == 'privada'")["Q022"].value_counts()
label_priv = q022_priv.index
value_priv = q022_priv.values

graph_q022 = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'},
                                          {'type':'domain'}]],
                  subplot_titles=["Escolas Públicas", "Escolas Privadas"])

graph_q022.add_trace(go.Pie(labels=label_pub, values=value_pub), 1, 1)
graph_q022.add_trace(go.Pie(labels=label_priv, values=value_priv), 1, 2)

graph_q022.update_layout(title_text=quest["Q022"])
graph_q022.update_layout(uniformtext_minsize=14, uniformtext_mode='hide')

Se observarmos para a maior ocorrência de pessoas por residência de escolas privadas, vemos que em 37,7% moram 4 pessoas, olhando para o número de aparelhos celulares por residência, isso daria pelo menos um celular por pessoa. Na maioria das residências de vestibulandos de escolas públicas também residem 4 pessoas, porém, não se conta com mesma proporção de celulares por residente.

In [ ]:
q024_pub = df_plot.query("TP_ESCOLA == 'publica'")["Q024"].value_counts()
label_pub = q024_pub.index
value_pub = q024_pub.values

q024_priv = df_plot.query("TP_ESCOLA == 'privada'")["Q024"].value_counts()
label_priv = q024_priv.index
value_priv = q024_priv.values

graph_q024 = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'},
                                          {'type':'domain'}]],
                  subplot_titles=["Escolas Públicas", "Escolas Privadas"])

graph_q024.add_trace(go.Pie(labels=label_pub, values=value_pub), 1, 1)
graph_q024.add_trace(go.Pie(labels=label_priv, values=value_priv), 1, 2)

graph_q024.update_layout(title_text=quest["Q024"])
graph_q024.update_layout(uniformtext_minsize=14, uniformtext_mode='hide')

Um item fundamental para qualquer estudante não está presente para 64,7% dos vestibulandos de escolas públicas. Enquanto que, para os de escolas privadas, 85,6% das residências possuem ao menos um aparelho.

In [ ]:
q025_pub = df_plot.query("TP_ESCOLA == 'publica'")["Q025"].value_counts()
label_pub = q025_pub.index
value_pub = q025_pub.values

q025_priv = df_plot.query("TP_ESCOLA == 'privada'")["Q025"].value_counts()
label_priv = q025_priv.index
value_priv = q025_priv.values

graph_q025 = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'},
                                          {'type':'domain'}]],
                  subplot_titles=["Escolas Públicas", "Escolas Privadas"])

graph_q025.add_trace(go.Pie(labels=label_pub, values=value_pub), 1, 1)
graph_q025.add_trace(go.Pie(labels=label_priv, values=value_priv), 1, 2)

graph_q025.update_layout(title_text=quest["Q025"])
graph_q025.update_layout(uniformtext_minsize=14, uniformtext_mode='hide')